# Hybrid HDB Price Prediction (Model Selection by MAPE) — v2

**Improvements over v1:**
1. **Time-aware train/test split** — split by transaction year instead of random, to prevent future-price leakage.
2. **Address-level target encoding** — block-level mean price added as a feature.
3. **TimeSeriesSplit cross-validation** for hyperparameter tuning instead of a single holdout.
4. **Stacked ensemble** — blends top-3 models (LightGBM, XGBoost, ExtraTrees) via Ridge meta-learner.
5. **Non-linear storey bins** — `level_band` (low/mid/high/penthouse) added alongside `level_mid`.
6. **Non-linear lease decay** — `lease_band` bins capture the sharp drop below 60 years.

Reads engineered features from `02_feature_layer/training/outputs` and writes model artefacts to `03_ml_layer/training/outputs`.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor, StackingRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler, TimeSeriesSplit, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

SEED = 42

#ROOT = Path('/Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens')
ROOT = Path('E:/Rekha/Learning/Meaching_Learning/Practise Module/VS Code PropertyLens')
FEATURE_OUTPUT = ROOT / '02_feature_layer' / 'training' / 'outputs'
ML_OUTPUT = ROOT / '03_ml_layer' / 'training' / 'outputs'
ML_OUTPUT.mkdir(parents=True, exist_ok=True)

RUN_DATE = datetime.now().strftime('%Y%m%d')

print('FEATURE_OUTPUT:', FEATURE_OUTPUT)
print('ML_OUTPUT:', ML_OUTPUT)

FEATURE_OUTPUT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/02_feature_layer/training/outputs
ML_OUTPUT: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/03_ml_layer/training/outputs


In [6]:
def latest_file(folder: Path, pattern: str) -> Path:
    files = sorted(folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No file for pattern {pattern} in {folder}')
    return files[-1]

train_fp = latest_file(FEATURE_OUTPUT, 'hdb_feature_train_*.csv')
test_fp  = latest_file(FEATURE_OUTPUT, 'hdb_feature_test_*.csv')

train_df = pd.read_csv(train_fp)
test_df  = pd.read_csv(test_fp)

# ── IMPROVEMENT 5: Non-linear storey bands ─────────────────────────────────
def add_level_band(df: pd.DataFrame) -> pd.DataFrame:
    """Bin floor level into ordinal bands that capture the non-linear premium."""
    if 'level_mid' not in df.columns:
        return df
    bins   = [0, 3, 10, 20, float('inf')]
    labels = [0, 1, 2, 3]  # low / mid / high / penthouse — kept as int for tree models
    df = df.copy()
    df['level_band'] = pd.cut(df['level_mid'], bins=bins, labels=labels, right=True).astype(float)
    return df

# ── IMPROVEMENT 6: Non-linear lease decay bands ────────────────────────────
def add_lease_band(df: pd.DataFrame) -> pd.DataFrame:
    """Bin remaining lease into bands; <60 yrs triggers a sharper price decay."""
    if 'lease_remaining_years' not in df.columns:
        return df
    bins   = [0, 40, 60, 75, float('inf')]
    labels = [0, 1, 2, 3]  # critical / short / standard / fresh
    df = df.copy()
    df['lease_band'] = pd.cut(df['lease_remaining_years'], bins=bins, labels=labels, right=True).astype(float)
    return df

train_df = add_level_band(add_lease_band(train_df))
test_df  = add_level_band(add_lease_band(test_df))

# ── IMPROVEMENT 1: Time-aware split ────────────────────────────────────────
# Instead of random 80/20 split, split strictly by transaction_year so the
# model never sees future prices during training — avoids temporal leakage.
if 'transaction_year' in train_df.columns and 'transaction_year' in test_df.columns:
    # Merge train+test, re-split by year boundary
    full_df = pd.concat([train_df, test_df], ignore_index=True)
    split_year = int(np.percentile(full_df['transaction_year'].dropna(), 80))
    train_df = full_df[full_df['transaction_year'] <= split_year].copy()
    test_df  = full_df[full_df['transaction_year'] >  split_year].copy()
    print(f'Time-aware split: train up to {split_year}, test after {split_year}')
else:
    print('transaction_year not found — falling back to original random split files')

target    = 'resale_price'
drop_cols = ['address_key'] if 'address_key' in train_df.columns else []

# ── IMPROVEMENT 2: Address-level target encoding ───────────────────────────
# Add block-level mean resale price from the TRAINING set only (no leakage).
if 'address_key' in train_df.columns:
    addr_mean = (
        train_df.groupby('address_key')[target]
        .mean()
        .rename('addr_mean_price')
        .reset_index()
    )
    global_mean = float(train_df[target].mean())
    train_df = train_df.merge(addr_mean, on='address_key', how='left')
    test_df  = test_df.merge(addr_mean, on='address_key', how='left')
    # Unseen addresses in test fall back to global training mean
    train_df['addr_mean_price'] = train_df['addr_mean_price'].fillna(global_mean)
    test_df['addr_mean_price']  = test_df['addr_mean_price'].fillna(global_mean)
    # Remove address_key from drop_cols since it's been used; the column itself still dropped below
    print(f'Address target encoding added (global fallback mean = {global_mean:,.0f})')

X_train = train_df.drop(columns=[target] + drop_cols)
y_train = train_df[target].values
X_test  = test_df.drop(columns=[target] + drop_cols)
y_test  = test_df[target].values

print('Train shape:', X_train.shape, '  Test shape:', X_test.shape)
print('Target range:', float(np.min(y_train)), 'to', float(np.max(y_train)))
print('New engineered features added: level_band, lease_band, addr_mean_price')

In [7]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)
    row = {
        'model':      name,
        'train_mape': float(mean_absolute_percentage_error(y_tr, pred_tr)),
        'test_mape':  float(mean_absolute_percentage_error(y_te, pred_te)),
        'test_mae':   float(mean_absolute_error(y_te, pred_te)),
        'test_rmse':  rmse(y_te, pred_te),
        'test_r2':    float(r2_score(y_te, pred_te)),
    }
    return row, pred_te

# ── IMPROVEMENT 3: TimeSeriesSplit tuning ──────────────────────────────────
# Sort the tuning sample by transaction_year so CV folds respect time order.
# This prevents the tuner from picking params that exploit future data.
def tune_by_tscv(model_cls, param_dist, X_fit, y_fit, n_splits=4, n_iter=12):
    """Select best params using TimeSeriesSplit CV scored by MAPE."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best_score  = float('inf')
    best_params = None
    for params in ParameterSampler(param_dist, n_iter=n_iter, random_state=SEED):
        fold_mapes = []
        for tr_idx, val_idx in tscv.split(X_fit):
            X_tr, X_val = X_fit.iloc[tr_idx], X_fit.iloc[val_idx]
            y_tr, y_val = y_fit[tr_idx], y_fit[val_idx]
            m = model_cls(**params)
            m.fit(X_tr, y_tr)
            fold_mapes.append(float(mean_absolute_percentage_error(y_val, m.predict(X_val))))
        cv_mape = float(np.mean(fold_mapes))
        if cv_mape < best_score:
            best_score  = cv_mape
            best_params = params
    return best_params, best_score

def build_refined_lgb_space(base):
    lr     = float(base['learning_rate'])
    leaves = int(base['num_leaves'])
    mc     = int(base['min_child_samples'])
    ne     = int(base['n_estimators'])
    return {
        'n_estimators':      sorted(set([max(250, ne - 150), ne, ne + 150])),
        'num_leaves':        sorted(set([max(16, leaves - 16), leaves, leaves + 16])),
        'learning_rate':     sorted(set([max(0.02, lr * 0.8), lr, min(0.1, lr * 1.2)])),
        'subsample':         sorted(set([max(0.7, float(base['subsample']) - 0.1), float(base['subsample']), min(1.0, float(base['subsample']) + 0.1)])),
        'colsample_bytree':  sorted(set([max(0.6, float(base['colsample_bytree']) - 0.1), float(base['colsample_bytree']), min(1.0, float(base['colsample_bytree']) + 0.1)])),
        'min_child_samples': sorted(set([max(10, mc - 15), mc, mc + 15])),
        'reg_lambda':        sorted(set([max(0.0, float(base['reg_lambda']) * 0.5), float(base['reg_lambda']), float(base['reg_lambda']) * 1.5 + 1e-9])),
        'objective':         ['regression'],
        'random_state':      [SEED],
        'n_jobs':            [-1],
        'verbose':           [-1],
    }

def build_refined_xgb_space(base):
    lr    = float(base['learning_rate'])
    depth = int(base['max_depth'])
    ne    = int(base['n_estimators'])
    mcw   = int(base['min_child_weight'])
    return {
        'n_estimators':     sorted(set([max(250, ne - 150), ne, ne + 150])),
        'max_depth':        sorted(set([max(4, depth - 1), depth, min(12, depth + 1)])),
        'learning_rate':    sorted(set([max(0.02, lr * 0.8), lr, min(0.12, lr * 1.2)])),
        'subsample':        sorted(set([max(0.7, float(base['subsample']) - 0.1), float(base['subsample']), min(1.0, float(base['subsample']) + 0.1)])),
        'colsample_bytree': sorted(set([max(0.5, float(base['colsample_bytree']) - 0.1), float(base['colsample_bytree']), min(1.0, float(base['colsample_bytree']) + 0.1)])),
        'min_child_weight': sorted(set([max(1, mcw - 1), mcw, mcw + 1])),
        'reg_lambda':       sorted(set([max(0.5, float(base['reg_lambda']) * 0.6), float(base['reg_lambda']), float(base['reg_lambda']) * 1.4])),
        'gamma':            sorted(set([max(0.0, float(base['gamma']) - 0.1), float(base['gamma']), float(base['gamma']) + 0.1])),
        'objective':        ['reg:squarederror'],
        'tree_method':      ['hist'],
        'random_state':     [SEED],
        'n_jobs':           [-1],
    }

# Use a capped sample for tuning speed; sort by year so TimeSeriesSplit folds are valid
max_tune_rows = 180000
if len(X_train) > max_tune_rows:
    rng      = np.random.default_rng(SEED)
    tune_idx = rng.choice(X_train.index.to_numpy(), size=max_tune_rows, replace=False)
    X_tune   = X_train.loc[tune_idx].copy()
    y_tune   = y_train[tune_idx]
else:
    X_tune = X_train.copy()
    y_tune = y_train.copy()

# Sort by transaction_year if present, so TSCV folds are chronologically ordered
if 'transaction_year' in X_tune.columns:
    sort_order = X_tune['transaction_year'].argsort().values
    X_tune = X_tune.iloc[sort_order].reset_index(drop=True)
    y_tune = y_tune[sort_order]

# Stage-1 broad random search (now with TimeSeriesSplit CV)
xgb_param_dist = {
    'n_estimators':     [300, 500, 700],
    'max_depth':        [5, 7, 9],
    'learning_rate':    [0.03, 0.05, 0.08],
    'subsample':        [0.75, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 6],
    'reg_lambda':       [1.0, 3.0, 8.0],
    'gamma':            [0.0, 0.1, 0.3],
    'objective':        ['reg:squarederror'],
    'tree_method':      ['hist'],
    'random_state':     [SEED],
    'n_jobs':           [-1],
}

lgb_param_dist = {
    'n_estimators':      [300, 500, 700],
    'num_leaves':        [31, 63, 127],
    'learning_rate':     [0.03, 0.05, 0.08],
    'subsample':         [0.75, 0.9, 1.0],
    'colsample_bytree':  [0.6, 0.8, 1.0],
    'min_child_samples': [20, 40, 80],
    'reg_lambda':        [0.0, 1.0, 5.0],
    'objective':         ['regression'],
    'random_state':      [SEED],
    'n_jobs':            [-1],
    'verbose':           [-1],
}

print('Running Stage-1 tuning with TimeSeriesSplit CV (n_splits=4) ...')
best_xgb_params_stage1, best_xgb_cv_mape_stage1 = tune_by_tscv(
    XGBRegressor, xgb_param_dist, X_tune, y_tune, n_splits=4, n_iter=12
)
best_lgb_params_stage1, best_lgb_cv_mape_stage1 = tune_by_tscv(
    LGBMRegressor, lgb_param_dist, X_tune, y_tune, n_splits=4, n_iter=12
)

# Stage-2 refined search
xgb_refined = build_refined_xgb_space(best_xgb_params_stage1)
lgb_refined = build_refined_lgb_space(best_lgb_params_stage1)

print('Running Stage-2 focused tuning ...')
best_xgb_params_stage2, best_xgb_cv_mape_stage2 = tune_by_tscv(
    XGBRegressor, xgb_refined, X_tune, y_tune, n_splits=4, n_iter=10
)
best_lgb_params_stage2, best_lgb_cv_mape_stage2 = tune_by_tscv(
    LGBMRegressor, lgb_refined, X_tune, y_tune, n_splits=4, n_iter=10
)

print('Stage1 XGBoost CV MAPE:', best_xgb_cv_mape_stage1)
print('Stage2 XGBoost CV MAPE:', best_xgb_cv_mape_stage2)
print('Stage1 LightGBM CV MAPE:', best_lgb_cv_mape_stage1)
print('Stage2 LightGBM CV MAPE:', best_lgb_cv_mape_stage2)

# ── Base models (retrain on full X_train after tuning) ─────────────────────
base_lgb = LGBMRegressor(**best_lgb_params_stage2)
base_xgb = XGBRegressor(**best_xgb_params_stage2)
base_et  = ExtraTreesRegressor(
    n_estimators=450, max_depth=None, min_samples_leaf=1,
    random_state=SEED, n_jobs=-1,
)

models = {
    'elastic_net_log_target': TransformedTargetRegressor(
        regressor=Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(alpha=0.001, l1_ratio=0.35, random_state=SEED, max_iter=30000)),
        ]),
        func=np.log1p, inverse_func=np.expm1,
    ),
    'random_forest': RandomForestRegressor(
        n_estimators=350, max_depth=24, min_samples_leaf=2,
        random_state=SEED, n_jobs=-1,
    ),
    'extra_trees':            base_et,
    'hist_gradient_boosting': HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=700, max_depth=10,
        min_samples_leaf=30, random_state=SEED,
    ),
    'xgboost_tuned_v2':   base_xgb,
    'lightgbm_tuned_v2':  base_lgb,
}

results    = []
pred_store = {}
for name, m in models.items():
    row, pred = evaluate(name, m, X_train, y_train, X_test, y_test)
    results.append(row)
    pred_store[name] = pred
    print(f"{name}: test_mape={row['test_mape']:.4f}  r2={row['test_r2']:.4f}")

score_df = pd.DataFrame(results).sort_values('test_mape').reset_index(drop=True)
score_df

In [ ]:
# ── IMPROVEMENT 4: Stacked Ensemble ───────────────────────────────────────
# Blend the top-3 base models (LightGBM, XGBoost, ExtraTrees) using a Ridge
# meta-learner.  StackingRegressor uses internal CV to build out-of-fold
# predictions for the meta-learner, so it does not overfit to the training set.

print('Training stacked ensemble (this may take a few minutes) ...')

stacking_model = StackingRegressor(
    estimators=[
        ('lgb', LGBMRegressor(**best_lgb_params_stage2)),
        ('xgb', XGBRegressor(**best_xgb_params_stage2)),
        ('et',  ExtraTreesRegressor(
            n_estimators=450, max_depth=None, min_samples_leaf=1,
            random_state=SEED, n_jobs=-1,
        )),
    ],
    final_estimator=Ridge(alpha=1.0),
    cv=5,          # 5-fold internal CV for generating meta-features
    n_jobs=-1,
    passthrough=False,
)

stacking_model.fit(X_train, y_train)
stack_pred_tr = stacking_model.predict(X_train)
stack_pred_te = stacking_model.predict(X_test)

stack_row = {
    'model':      'stacking_lgb_xgb_et',
    'train_mape': float(mean_absolute_percentage_error(y_train, stack_pred_tr)),
    'test_mape':  float(mean_absolute_percentage_error(y_test, stack_pred_te)),
    'test_mae':   float(mean_absolute_error(y_test, stack_pred_te)),
    'test_rmse':  float(np.sqrt(mean_squared_error(y_test, stack_pred_te))),
    'test_r2':    float(r2_score(y_test, stack_pred_te)),
}
pred_store['stacking_lgb_xgb_et'] = stack_pred_te

# Add stacking to results and re-rank
results.append(stack_row)
score_df = pd.DataFrame(results).sort_values('test_mape').reset_index(drop=True)
models['stacking_lgb_xgb_et'] = stacking_model

print(f"Stacking ensemble: test_mape={stack_row['test_mape']:.4f}  r2={stack_row['test_r2']:.4f}")
score_df

In [8]:
best_name  = score_df.iloc[0]['model']
best_model = models[best_name]
# Re-fit best model on full training data (StackingRegressor is already fitted;
# for single models we refit to make sure they used full X_train)
if not isinstance(best_model, StackingRegressor):
    best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print('Best model by MAPE:', best_name)
print('Best test MAPE:', float(score_df.iloc[0]['test_mape']))

pred_df = test_df.copy()
pred_df['predicted_price']  = best_pred
pred_df['abs_pct_error']    = np.abs((pred_df['resale_price'] - pred_df['predicted_price']) / pred_df['resale_price'])

score_fp = ML_OUTPUT / f'model_mape_leaderboard_{RUN_DATE}.csv'
pred_fp  = ML_OUTPUT / f'best_model_predictions_{RUN_DATE}.csv'
model_fp = ML_OUTPUT / f'best_price_model_{RUN_DATE}.joblib'
meta_fp  = ML_OUTPUT / f'model_metadata_{RUN_DATE}.json'

score_df.to_csv(score_fp, index=False)
pred_df.to_csv(pred_fp,   index=False)
joblib.dump(best_model,    model_fp)

metadata = {
    'run_date':       RUN_DATE,
    'train_file':     str(train_fp),
    'test_file':      str(test_fp),
    'best_model':     best_name,
    'best_test_mape': float(score_df.iloc[0]['test_mape']),
    'improvements':   [
        'time_aware_split',
        'address_target_encoding',
        'timeseries_cv_tuning',
        'stacking_ensemble',
        'level_band_feature',
        'lease_band_feature',
    ],
    'metrics':        score_df.to_dict(orient='records'),
    'artifacts': {
        'leaderboard':  str(score_fp),
        'predictions':  str(pred_fp),
        'model':        str(model_fp),
    },
}

with open(meta_fp, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved leaderboard:', score_fp.name)
print('Saved predictions:', pred_fp.name)
print('Saved model:',       model_fp.name)
print('Saved metadata:',    meta_fp.name)

In [9]:
# Feature importance comparison for key tree models
importance_targets = ['extra_trees', 'lightgbm_tuned_v2', 'xgboost_tuned_v2']
feat_names = X_train.columns.tolist()
imp_frames = []

for model_name in importance_targets:
    if model_name not in models:
        continue
    m = models[model_name]
    importances = getattr(m, 'feature_importances_', None)
    if importances is None:
        continue

    df_imp = pd.DataFrame({
        'feature': feat_names,
        model_name: importances,
    })
    df_imp[model_name] = pd.to_numeric(df_imp[model_name], errors='coerce').fillna(0.0)
    total = df_imp[model_name].sum()
    if total > 0:
        df_imp[model_name] = df_imp[model_name] / total
    imp_frames.append(df_imp)

if not imp_frames:
    print('No feature importances available from selected models.')
else:
    imp_all = imp_frames[0]
    for nxt in imp_frames[1:]:
        imp_all = imp_all.merge(nxt, on='feature', how='outer')

    for c in imp_all.columns:
        if c != 'feature':
            imp_all[c] = imp_all[c].fillna(0.0)

    model_cols = [c for c in imp_all.columns if c != 'feature']
    imp_all['importance_mean'] = imp_all[model_cols].mean(axis=1)
    imp_all = imp_all.sort_values('importance_mean', ascending=False).reset_index(drop=True)

    imp_fp = ML_OUTPUT / f'feature_importance_comparison_{RUN_DATE}.csv'
    imp_all.to_csv(imp_fp, index=False)

    print('Saved feature importance comparison:', imp_fp.name)
    imp_all.head(20)

Saved feature importance comparison: feature_importance_comparison_20260317.csv


In [12]:
# SHAP explanation for a user-input unit address
# Usage: update `unit_address_input`, run this cell, then inspect prediction + contributions.

import re

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from xgboost import DMatrix, XGBRegressor

import shap


def normalize_unit_address(addr: str) -> str:
    s = str(addr).upper().strip()
    s = re.sub(r',\s*SINGAPORE\s*\d*$', '', s)
    s = re.sub(r'\s+#\d{1,2}-\d{1,4}[A-Z]?\s*$', '', s)  # remove unit suffix
    s = re.sub(r'\s+', ' ', s)
    return s


def to_model_vector(row_df: pd.DataFrame, model_columns: list[str]) -> pd.DataFrame:
    x = row_df.reindex(columns=model_columns, fill_value=0)
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors='coerce').fillna(0)
    return x


def summarize_onehot_contributions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def bucket(feature_name: str) -> str:
        if feature_name.startswith('town_'):
            return 'town(one-hot)'
        if feature_name.startswith('flat_type_'):
            return 'flat_type(one-hot)'
        if feature_name.startswith('flat_model_'):
            return 'flat_model(one-hot)'
        return feature_name

    out['feature_group'] = out['feature'].map(bucket)
    g = out.groupby('feature_group', as_index=False).agg(
        contribution=('shap_value', 'sum'),
        abs_contribution=('abs_shap', 'sum')
    )
    g = g.sort_values('abs_contribution', ascending=False).reset_index(drop=True)
    return g


def best_address_fallback(query: str, all_addresses: pd.Series) -> str | None:
    tokens = [t for t in re.split(r'\s+', query) if len(t) >= 3]
    if not tokens:
        return None

    unique_addr = all_addresses.dropna().astype(str).drop_duplicates().reset_index(drop=True)
    scores = pd.Series(0, index=unique_addr.index, dtype='int64')
    for tok in tokens[:5]:
        scores = scores + unique_addr.str.contains(re.escape(tok), regex=True, na=False).astype(int)

    if scores.max() <= 0:
        return None
    return unique_addr.iloc[int(scores.idxmax())]


def get_tree_shap_values(model, x_row_df: pd.DataFrame) -> tuple[np.ndarray, float]:
    # Prefer native TreeSHAP implementations for stability and speed.
    if isinstance(model, LGBMRegressor) and hasattr(model, 'booster_'):
        contrib = model.booster_.predict(x_row_df, pred_contrib=True)
        # LightGBM returns [feature_contribs..., bias]
        shap_values = np.array(contrib[0][:-1], dtype=float)
        base_value = float(contrib[0][-1])
        return shap_values, base_value

    if isinstance(model, XGBRegressor):
        dm = DMatrix(x_row_df, feature_names=list(x_row_df.columns))
        contrib = model.get_booster().predict(dm, pred_contribs=True)
        # XGBoost returns [feature_contribs..., bias]
        shap_values = np.array(contrib[0][:-1], dtype=float)
        base_value = float(contrib[0][-1])
        return shap_values, base_value

    # Fallback for other tree models
    explainer = shap.TreeExplainer(model, feature_perturbation='tree_path_dependent')
    sv = explainer.shap_values(x_row_df)
    if isinstance(sv, list):
        sv = sv[0]
    shap_values = np.array(sv).reshape(-1)

    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        base_value = float(np.array(expected_value).reshape(-1)[0])
    else:
        base_value = float(expected_value)
    return shap_values, base_value


# 1) Input a detailed address with unit number
unit_address_input = '174 ANG MO KIO AVE 4 #05-123'

# 2) Load the latest engineered feature table (contains address_key + encoded features)
feature_table_fp = latest_file(FEATURE_OUTPUT, 'hdb_feature_table_*.csv')
feature_df = pd.read_csv(feature_table_fp)

if 'address_key' not in feature_df.columns:
    raise ValueError('Feature table does not contain address_key, cannot match unit address.')

# 3) Normalize and match candidates
query = normalize_unit_address(unit_address_input)
feature_df['address_key_norm'] = feature_df['address_key'].astype(str).str.upper().str.replace(r'\s+', ' ', regex=True).str.strip()

exact_mask = feature_df['address_key_norm'] == query
candidate = feature_df[exact_mask].copy()
matched_mode = 'exact'

if candidate.empty:
    # relaxed match: all meaningful tokens appear in address_key
    tokens = [t for t in re.split(r'\s+', query) if len(t) >= 3]
    if tokens:
        relaxed_mask = pd.Series(True, index=feature_df.index)
        for tok in tokens[:4]:  # cap to keep match stable/fast
            relaxed_mask = relaxed_mask & feature_df['address_key_norm'].str.contains(re.escape(tok), regex=True, na=False)
        candidate = feature_df[relaxed_mask].copy()
        matched_mode = 'relaxed'

if candidate.empty:
    fallback_addr = best_address_fallback(query, feature_df['address_key_norm'])
    if fallback_addr is None:
        sample_keys = feature_df['address_key'].dropna().astype(str).head(10).tolist()
        raise ValueError(
            'No address matched and no fallback candidate found. '
            f'Please adjust address format. Example keys: {sample_keys}'
        )
    candidate = feature_df[feature_df['address_key_norm'] == fallback_addr].copy()
    matched_mode = 'fallback_best_similarity'

# Prefer latest transaction record for that address
if 'transaction_year' in candidate.columns:
    candidate = candidate.sort_values('transaction_year', ascending=False)
row = candidate.iloc[[0]].copy()

# 4) Build model input and predict
if 'best_model' in globals() and 'best_name' in globals():
    model_for_infer = best_model
    model_name_for_infer = best_name
else:
    model_fp_latest = latest_file(ML_OUTPUT, 'best_price_model_*.joblib')
    model_for_infer = joblib.load(model_fp_latest)
    model_name_for_infer = model_fp_latest.stem

model_cols = X_train.columns.tolist()
x_row = to_model_vector(row, model_cols)
pred_price = float(model_for_infer.predict(x_row)[0])

# 5) SHAP explanation (native TreeSHAP where possible)
shap_values, base_value = get_tree_shap_values(model_for_infer, x_row)

contrib_df = pd.DataFrame({
    'feature': model_cols,
    'feature_value': x_row.iloc[0].values,
    'shap_value': shap_values,
})
contrib_df['abs_shap'] = contrib_df['shap_value'].abs()
contrib_df = contrib_df.sort_values('abs_shap', ascending=False).reset_index(drop=True)

contrib_group_df = summarize_onehot_contributions(contrib_df)

# 6) Save outputs
safe_addr = re.sub(r'[^A-Z0-9]+', '_', str(row['address_key'].iloc[0]))[:60]
raw_fp = ML_OUTPUT / f'shap_contribution_raw_{RUN_DATE}_{safe_addr}.csv'
group_fp = ML_OUTPUT / f'shap_contribution_grouped_{RUN_DATE}_{safe_addr}.csv'
contrib_df.to_csv(raw_fp, index=False)
contrib_group_df.to_csv(group_fp, index=False)

# 7) Show summary
print('Address input:', unit_address_input)
print('Match mode:', matched_mode)
print('Matched address_key:', str(row['address_key'].iloc[0]))
print('Model used:', model_name_for_infer)
print('Predicted resale price:', round(pred_price, 2))
print('SHAP base value:', round(base_value, 2))
print('Top 15 feature contributions:')
display(contrib_df[['feature', 'feature_value', 'shap_value']].head(15))
print('Top grouped contributions:')
display(contrib_group_df.head(12))
print('Saved raw SHAP:', raw_fp.name)
print('Saved grouped SHAP:', group_fp.name)

Address input: 174 ANG MO KIO AVE 4 #05-123
Match mode: exact
Matched address_key: 174 ANG MO KIO AVE 4
Model used: lightgbm_tuned_v2
Predicted resale price: 438169.62
SHAP base value: 512309.67
Top 15 feature contributions:


,feature,feature_value,shap_value
0,transaction_year,2026,120653.669400
1,floor_area_sqm,69.0,-98007.484166
2,room_count,3.0,-59895.178995
3,mall_weighted_access_3km,0.435691,-11391.273947
4,dist_to_nearest_mall_m,2045.204033,11311.505531
5,town_QUEENSTOWN,False,-6519.497411
6,mall_count_3km,1,-5841.827224
7,dist_to_foodcourt_m,1040.502729,-4486.034692
8,flat_type_5 ROOM,False,4138.238674
9,school_count_1km,7,-3597.369257


Top grouped contributions:


,feature_group,contribution,abs_contribution
0,transaction_year,120653.669400,120653.669400
1,floor_area_sqm,-98007.484166,98007.484166
2,room_count,-59895.178995,59895.178995
3,town(one-hot),-14189.267586,22655.120050
4,mall_weighted_access_3km,-11391.273947,11391.273947
5,dist_to_nearest_mall_m,11311.505531,11311.505531
6,flat_type(one-hot),4203.164290,9281.027167
7,mall_count_3km,-5841.827224,5841.827224
8,dist_to_foodcourt_m,-4486.034692,4486.034692
9,flat_model(one-hot),-1512.136591,4027.806873


Saved raw SHAP: shap_contribution_raw_20260317_174_ANG_MO_KIO_AVE_4.csv
Saved grouped SHAP: shap_contribution_grouped_20260317_174_ANG_MO_KIO_AVE_4.csv


In [13]:
# School-factor contribution focus (always show these factors)
school_features = [
    'dist_to_nearest_school_m',
    'school_count_1km',
    'primary_school_quality_1km_weighted',
    'primary_school_top_quality_1km',
    'primary_school_count_1km',
]

missing_school_features = [f for f in school_features if f not in contrib_df['feature'].values]
if missing_school_features:
    print('Missing school features in model input:', missing_school_features)

school_detail = contrib_df[contrib_df['feature'].isin(school_features)].copy()
school_detail = school_detail[['feature', 'feature_value', 'shap_value', 'abs_shap']].sort_values('abs_shap', ascending=False)

print('School-related feature contributions:')
display(school_detail)

school_group = contrib_group_df[contrib_group_df['feature_group'].isin(school_features)].copy()
school_group_total = pd.DataFrame({
    'feature_group': ['school_all_factors'],
    'contribution': [school_group['contribution'].sum()],
    'abs_contribution': [school_group['abs_contribution'].sum()],
})

print('School grouped contributions:')
display(pd.concat([school_group, school_group_total], ignore_index=True))

School-related feature contributions:


,feature,feature_value,shap_value,abs_shap
9,school_count_1km,7,-3597.369257,3597.369257
26,primary_school_quality_1km_weighted,14.134463,-756.201408,756.201408
30,primary_school_count_1km,2,-467.488548,467.488548
46,dist_to_nearest_school_m,415.256391,-90.842757,90.842757
48,primary_school_top_quality_1km,14.134463,-72.942979,72.942979


School grouped contributions:


,feature_group,contribution,abs_contribution
0,school_count_1km,-3597.369257,3597.369257
1,primary_school_quality_1km_weighted,-756.201408,756.201408
2,primary_school_count_1km,-467.488548,467.488548
3,dist_to_nearest_school_m,-90.842757,90.842757
4,primary_school_top_quality_1km,-72.942979,72.942979
5,school_all_factors,-4984.844949,4984.844949


In [14]:
# Full feature dictionary: all model features + descriptions
feature_list = X_train.columns.tolist()

def describe_feature(feat: str) -> tuple[str, str]:
    # returns: (factor_category, description_cn)
    static_desc = {
        'transaction_year': ('time', '交易年份，用于捕捉市场周期与通胀趋势'),
        'level_mid': ('property_physical', '楼层中位值，反映高低楼层差异'),
        'lease_remaining_years': ('property_physical', '剩余租约年限（99年制下的剩余年数）'),
        'floor_area_sqm': ('property_physical', '套内面积（平方米）'),
        'room_count': ('property_physical', '房间数（由房型解析）'),
        'dist_to_mrt_m': ('accessibility', '到最近地铁站距离（米）'),
        'orientation_score': ('environment', '朝向/临路得分（临路扣分，非临路加分）'),
        'dist_to_highway_m': ('environment', '到高速公路距离（米），越近噪音与污染潜在影响越大'),
        'dist_to_foodcourt_m': ('amenities', '到最近食阁/小贩中心距离（米）'),
        'dist_to_nearest_mall_m': ('amenities', '到最近商业体距离（米）'),
        'mall_count_3km': ('amenities', '3公里范围内商业体数量'),
        'mall_weighted_access_3km': ('amenities', '3公里商业体加权可达性（数量、规模、距离联合）'),
        'dist_to_nearest_school_m': ('school', '到最近学校直线距离（米）'),
        'school_count_1km': ('school', '1公里内学校数量（全学段）'),
        'primary_school_quality_1km_weighted': ('school', '1公里内小学质量加权得分（由报名竞争度反推）'),
        'primary_school_top_quality_1km': ('school', '1公里内最高小学质量得分'),
        'primary_school_count_1km': ('school', '1公里内小学数量'),
    }

    if feat in static_desc:
        return static_desc[feat]
    if feat.startswith('town_'):
        return ('location_onehot', '所在镇区哑变量（该镇区=1，否则=0）')
    if feat.startswith('flat_type_'):
        return ('flat_type_onehot', '房型哑变量（该房型=1，否则=0）')
    if feat.startswith('flat_model_'):
        return ('flat_model_onehot', '房屋模型哑变量（该模型=1，否则=0）')
    return ('other', '特征说明待补充（请检查特征工程输出）')

rows = []
for f in feature_list:
    category, desc = describe_feature(f)
    rows.append({
        'feature': f,
        'factor_category': category,
        'description_cn': desc,
    })

feature_dict_df = pd.DataFrame(rows)
feature_dict_df = feature_dict_df.sort_values(['factor_category', 'feature']).reset_index(drop=True)

feature_dict_fp = ML_OUTPUT / f'feature_dictionary_{RUN_DATE}.csv'
feature_dict_df.to_csv(feature_dict_fp, index=False)

print('Total model features:', len(feature_list))
print('Saved feature dictionary:', feature_dict_fp.name)
print('Feature count by category:')
display(feature_dict_df.groupby('factor_category', as_index=False).size().sort_values('size', ascending=False))
print('Feature dictionary preview:')
display(feature_dict_df.head(40))

Total model features: 69
Saved feature dictionary: feature_dictionary_20260317.csv
Feature count by category:


,factor_category,size
5,location_onehot,25
3,flat_model_onehot,21
4,flat_type_onehot,6
7,school,5
1,amenities,4
6,property_physical,4
2,environment,2
0,accessibility,1
8,time,1


Feature dictionary preview:


,feature,factor_category,description_cn
0,dist_to_mrt_m,accessibility,到最近地铁站距离（米）
1,dist_to_foodcourt_m,amenities,到最近食阁/小贩中心距离（米）
2,dist_to_nearest_mall_m,amenities,到最近商业体距离（米）
3,mall_count_3km,amenities,3公里范围内商业体数量
4,mall_weighted_access_3km,amenities,3公里商业体加权可达性（数量、规模、距离联合）
5,dist_to_highway_m,environment,到高速公路距离（米），越近噪音与污染潜在影响越大
6,orientation_score,environment,朝向/临路得分（临路扣分，非临路加分）
7,flat_model_3Gen,flat_model_onehot,房屋模型哑变量（该模型=1，否则=0）
8,flat_model_Adjoined flat,flat_model_onehot,房屋模型哑变量（该模型=1，否则=0）
9,flat_model_Apartment,flat_model_onehot,房屋模型哑变量（该模型=1，否则=0）
